In [56]:
from kag.solver.logic.solver_pipeline import SolverPipeline
from kag.common.conf import KAG_CONFIG
from kag.common.registry import import_modules_from_path
import pandas as pd 
import_modules_from_path("./prompt")

OPENAI_API_KEY='sk-yTJxTqRrqRdP0DKDqwPxDyHHE149jcis5x4qFwOUR9mIaP6D'

#  假设question 

In [39]:
question = """how to make sure the coolant refill container is filled with 5 gallons of coolant concentrate"""

# 1.从failure type检索

In [17]:
from langchain_community.embeddings import OpenAIEmbeddings
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(
         openai_api_key=OPENAI_API_KEY,
        base_url="https://api.chatanywhere.com.cn/v1",
)

index = faiss.IndexFlatL2(len(embeddings.embed_query("hello world")))

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

INFO:faiss.loader:Loading faiss with AVX2 support.
INFO:faiss.loader:Successfully loaded faiss with AVX2 support.
INFO:faiss:Failed to load GPU Faiss: name 'GpuIndexIVFFlat' is not defined. Will not load constructor refs for GPU indexes. This is only an error if you're trying to use GPU Faiss.


In [13]:
failure_type = pd.read_excel("../../../builder/repair_feedback.xlsx")   

In [20]:
from uuid import uuid4
uuids = [str(uuid4()) for _ in range(len(failure_type['operator_solution'].tolist()))]

vector_store.add_texts(texts=failure_type['operator_solution'].tolist(), ids=uuids)

['fb844325-73cc-45d4-85d0-ff81b9eaa4b0',
 '9e0b3f48-b7d5-4ee3-b52b-8af036629279',
 'd2fdd7b7-562b-4a73-a992-7c3be6921ecb',
 'a4d3a32f-7a15-4e83-818f-fa4028f83b6a',
 '09c0a14f-e6bc-4b67-8264-241968b61239',
 'ac7a7726-acfe-433b-aac6-200d4dc90999',
 'a78aaaff-623b-484a-a21e-1d1bd93486f9',
 'c310a262-468d-48ab-af45-57815348523e',
 'f76458bc-e862-454a-a788-3ede642bbf2d',
 '418c2c3b-8168-40a0-85a7-e2596998bf8b',
 '9f35eaba-3483-464a-9a57-a4133ea06bcd',
 '3054afae-d52a-4201-ae6f-46802016ac2a',
 'c3108dee-5055-472b-ae78-5e39d4230dec',
 'd397264f-e1dc-4b83-87f6-e9c4da48e870',
 '0ba2dc3e-3323-4971-8a43-a45e41fc4e03',
 '60677b27-95a9-4ede-a7a1-a825a3187cb7',
 'c68b46c6-ca03-46d1-9cf3-a0412354a735',
 'a3bd19b8-d94d-4b0b-98ef-045d80889729',
 '63a78353-a513-4ab7-8e75-f9eaa8523ee8',
 '6cc04d88-5f71-4fbd-a29c-f6f1c164906b',
 '4e38d57d-fffb-41e1-8087-3aabfbdd6973',
 '56787ddf-0b77-40ab-b3c8-e208bee62ba8',
 'ac7e36ef-d273-4b86-88ef-1180130fb81c',
 '5d7b0f87-a769-4828-8cf4-fb6d1603421e',
 '4ea96abb-dbd7-

In [64]:
vector_store.save_local("./faiss_index")

In [43]:
event_log_answer = vector_store.similarity_search(question,k=1) 

In [44]:
event_log_answer[0].page_content

'Set the maximum coolant using the remaining concentrate.'

# 2.从kag检索

In [45]:
def qa(query):
    resp = SolverPipeline.from_config(KAG_CONFIG.all_config["kag_solver_pipeline"])
    answer, traceLog = resp.run(query)
    return answer, traceLog

In [46]:
answer, traceLog = qa(question)

In [51]:
traceLog[0].keys()

dict_keys(['sub question', 'recall docs', 'rerank docs', 'kg_exact_solved_answer', 'present_instruction', 'present_memory'])

In [54]:
traceLog[0]['rerank docs']

['#Coolant Refill - Haas Service Manual#Coolant Refill - Haas Service Manual\nFITG BLKHD NPT3/8M/F X\n\nNPT3/4M X 2-1/2L\n\n23.\nFITG REDUCER NPT1-1/4M\n\nNPT3/4F NYLON\n\n24.\nGREASE STRAINER\n\n25.\n5G TANK\n\n## Introduction\n\nThe image show the following\ncomponents of the coolant refill\nsystem.\n1.\nCoolant concentrate tank\n\n2.\nStrainer\n\n3.\nSolenoid cable\n4.\nFiller hose\n\n5.\nWater solenoid cable\n\n6.',
 '#BMT65_75 - Turret Indexer Assembly - Troubleshooting Guide#BMT65_75 - Turret Indexer Assembly - Troubleshooting Guide\nWhen the machine has the correct air pressure and flow, the needle on\nthe air pressure gauge should not drop more than 10 PSI (0.\n70 bar) during a tool change.\nThis test confirms\nthat the air pressure and air flow into the machine are correct.',
 '#BMT65_75 - Turret Indexer Assembly - Troubleshooting Guide#BMT65_75 - Turret Indexer Assembly - Troubleshooting Guide\nFor Haas machine air\npressure specifications, go to New Machine Pre-Installation 

# 3 .将以上两个数据源合并给llm再回答

In [57]:
from openai import OpenAI
from langchain_community.vectorstores import FAISS

client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url="https://api.chatanywhere.tech/v1"
)

In [58]:
def generate_answer(question,traceLog,solution):
    response = client.chat.completions.create(
        model="deepseek-reasoner",
        messages=[{"role":"system","content":"""I will provide a machine malfunction issue, some similar cases found in the manual book, and solutions previously used by operators from the event log. Based on this information, please select the relevant parts and use them to answer the malfunction issue.
    
    #NOTE:
    1.Reply in English.
         """
                   },
    {"role":"user","content":f"machine malfunction issue：{question}\n"
                             f"similar cases found in the manual book：{traceLog}"
                            f"solutions previously used by operators from the event log：{solution}"}],
        temperature=0.01
    )
    return response.choices[0].message.content 

In [59]:
final_answer = generate_answer(question=question,solution=event_log_answer[0].page_content,traceLog=traceLog[0]['rerank docs'])

In [62]:
print(final_answer) 



To ensure the coolant refill container is filled with **5 gallons of coolant concentrate**, follow these steps based on the provided manual excerpts and operator solutions:  

1. **Verify the Tank Capacity**:  
   - The manual references a **5G TANK** (5-gallon tank) as part of the coolant refill system. Confirm this tank is designated for coolant concentrate.  

2. **Use the Filler Hose and Solenoid System**:  
   - The coolant refill system includes a **filler hose** and **solenoid components** (e.g., water solenoid cable). Ensure these are properly connected to automate the mixing and refill process.  

3. **Monitor via Control Settings**:  
   - Operators previously resolved similar issues by **"setting the maximum coolant using the remaining concentrate"**. Adjust the coolant concentration settings in the machine’s control interface to match the 5-gallon requirement.  

4. **Check for Obstructions**:  
   - Inspect the **strainer** (item 2 in the coolant refill system) for block

# concat

In [63]:
def get_final_answer(question):
    #kag
    answer, traceLog = qa(question)
    #event_log
    fund_names_db = FAISS.load_local(r"./faiss_index",
                                     embeddings,
                                     allow_dangerous_deserialization=True)
    event_log_answer = fund_names_db.similarity_search(question,k=1) 
    
    final_answer = generate_answer(question=question,solution=event_log_answer[0].page_content,traceLog=traceLog)
    return final_answer